<a href="https://colab.research.google.com/github/divyanshuraj25/Day_16_RAG_Failure_Diagnostics/blob/main/Day_16_RAG_Diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.5 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()

Saving document.pdf.pdf to document.pdf (1).pdf


In [4]:
import os

print(os.listdir())

['.config', 'document.pdf.pdf', 'document.pdf (1).pdf', 'sample_data']


In [5]:
import os
from pypdf import PdfReader

pdf_files = [f for f in os.listdir() if f.lower().endswith(".pdf")]

print("PDF files:", pdf_files)

pdf_path = pdf_files[0]

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("Total characters:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])

PDF files: ['document.pdf.pdf', 'document.pdf (1).pdf']
Total characters: 24147

First 1000 characters:

Unit – 1: Lec – 1
Today’s Target
 Need of Programming Languages
 Introduction of Python
 First Program in Python
 Identifiers, Keywords & Comments
 Variables 
 AKTU PYQs
Gateway Classes 
Gateway Classes 
UNIT-1: LEC-1 AKTU PYQs
Q.1 : Write a Python program to print the following Output :    Welcome to Python Programming.
Q.2 : Write a Python Program to print your :  Name, Branch, College Name using print().
Q.3 : What are Python Variables.
Q.4 : Create Variables to store :
• Student name
• Age
• Marks
Q.5 :  Identify the following Variable names are Invalid or Valid :
• 1age
• College_name
• Marks1
• Your name 
Gateway Classes 
WHY PROGRAMMING LANGUAGES WERE NEEDED?
• A computer cannot understand human language directly. It can understand only machine language, which 
is made up of 0s and 1s. Writing instructions in machine language is very difficult and time consuming 
for hu

In [6]:
chunk_size = 500
overlap = 100

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]

    if chunk.strip():
        chunks.append(chunk.strip())

    start += chunk_size - overlap

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:500])

Total chunks: 61

--- Chunk 1 ---
Unit – 1: Lec – 1
Today’s Target
 Need of Programming Languages
 Introduction of Python
 First Program in Python
 Identifiers, Keywords & Comments
 Variables 
 AKTU PYQs
Gateway Classes 
Gateway Classes 
UNIT-1: LEC-1 AKTU PYQs
Q.1 : Write a Python program to print the following Output :    Welcome to Python Programming.
Q.2 : Write a Python Program to print your :  Name, Branch, College Name using print().
Q.3 : What are Python Variables.
Q.4 : Create Variables to store :
• Student name
•

--- Chunk 2 ---
e using print().
Q.3 : What are Python Variables.
Q.4 : Create Variables to store :
• Student name
• Age
• Marks
Q.5 :  Identify the following Variable names are Invalid or Valid :
• 1age
• College_name
• Marks1
• Your name 
Gateway Classes 
WHY PROGRAMMING LANGUAGES WERE NEEDED?
• A computer cannot understand human language directly. It can understand only machine language, which 
is made up of 0s and 1s. Writing instructions in machine langua

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [8]:
import numpy as np

embeddings = model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (61, 384)


In [9]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print("FAISS index created!")
print("Total vectors:", index.ntotal)

FAISS index created!
Total vectors: 61


In [10]:
def retrieve_chunks(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        results.append({
            "rank": rank,
            "chunk_id": int(idx),
            "distance": float(distance),
            "content": chunks[idx]
        })

    return results

In [11]:
query = "What is machine learning?"

results = retrieve_chunks(query, top_k=3)

for result in results:
    print(f"\nRank: {result['rank']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Distance: {result['distance']:.4f}")
    print("Content:")
    print(result["content"][:500])


Rank: 1
Chunk ID: 6
Distance: 1.2542
Content:
uctions 
Output Displayed on the Screen
Converts Source 
code into byte code
Gateway Classes 
Importance Meaning 
1. Easy to Learn and Understand Python has simple and readable syntax, so beginners can learn 
programming easily.
Its language is close to English, which makes coding simple.
2. Used in Artificial Intelligence 
and Machine Learning
Python is widely used in AI and ML applications like chatbots, face 
recognition, and recommendation systems.
Many modern technologies are built using Py

Rank: 2
Chunk ID: 5
Distance: 1.3342
Content:
rtificial intelligence, 
machine learning, and automation.
• Python provides simple syntax and requires fewer lines of code, which makes programming easier and 
more efficient.
 Gateway Classes 
Source Code File 
File Name : program.py
Python Interpreter 
Reads the .py file line by line 
Byte Code Generation 
File Extension : .pyc
Python Virtual Machine (PVM) 
Converts byte code into machine instructio

In [12]:
test_queries = [
    # 1–3: Retrieval failure
    "What is the main conclusion of this document?",
    "What specific example is given about the final topic?",
    "What is the exact definition mentioned in the document?",

    # 4–6: Context / chunk boundary problems
    "How are the two concepts explained in relation to each other?",
    "What happens immediately after the process described in the document?",
    "What are the steps of the process in the correct order?",

    # 7–9: Vague or irrelevant retrieval
    "Tell me something important from this document.",
    "What is discussed in this section?",
    "What are the important points?",

    # 10–12: Answer-context mismatch
    "Why is this concept important according to the document?",
    "What advantage does the document mention?",
    "What limitation is described in the document?",

    # 13–15: Potential hallucination / unsupported questions
    "According to the document, what happened in 2025?",
    "What does the document say about quantum computing?",
    "Who is the author of this document?"
]

print("Total test queries:", len(test_queries))

for i, query in enumerate(test_queries, start=1):
    print(f"{i}. {query}")

Total test queries: 15
1. What is the main conclusion of this document?
2. What specific example is given about the final topic?
3. What is the exact definition mentioned in the document?
4. How are the two concepts explained in relation to each other?
5. What happens immediately after the process described in the document?
6. What are the steps of the process in the correct order?
7. Tell me something important from this document.
8. What is discussed in this section?
9. What are the important points?
10. Why is this concept important according to the document?
11. What advantage does the document mention?
12. What limitation is described in the document?
13. According to the document, what happened in 2025?
14. What does the document say about quantum computing?
15. Who is the author of this document?


In [13]:
all_results = []

for query in test_queries:
    results = retrieve_chunks(query, top_k=3)

    all_results.append({
        "query": query,
        "retrieved_chunks": results
    })

print("Queries processed:", len(all_results))

Queries processed: 15


In [14]:
for i, item in enumerate(all_results, start=1):
    print("\n" + "=" * 70)
    print(f"QUERY {i}: {item['query']}")

    for result in item["retrieved_chunks"]:
        print(f"\nRank {result['rank']} | Distance: {result['distance']:.4f}")
        print(result["content"][:300].replace("\n", " "))


QUERY 1: What is the main conclusion of this document?

Rank 1 | Distance: 1.6389
Unit – 1: Lec – 1 Today’s Target  Need of Programming Languages  Introduction of Python  First Program in Python  Identifiers, Keywords & Comments  Variables   AKTU PYQs Gateway Classes  Gateway Classes  UNIT-1: LEC-1 AKTU PYQs Q.1 : Write a Python program to print the following Output :    We

Rank 2 | Distance: 1.6839
: Write short notes on the following with examples : AKTU(2024-25) a) Operator Precedence b) Python Indentation c) Type Conversion  Q.6 : Differentiate between / and // operator with an example . AKTU(2023-24) Q.7 Explain the input () function with syntax & examples . Q.8 Differentiate between is an

Rank 3 | Distance: 1.7167
s .   Given : a = 3  b = 3 Gateway Classes  OPERATORS PRECEDENCE & ASSOCIATIVITY OPERATOR PRECEDENCE : • Operator Precedence defines the order in which operators are evaluated in an expression . • Operators with higher precedence are evaluated before operators 

In [15]:
import json

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("results.json created successfully!")

results.json created successfully!


In [16]:
import os

print("results.json exists:", os.path.exists("results.json"))
print("File size:", os.path.getsize("results.json"), "bytes")

results.json exists: True
File size: 30399 bytes


In [22]:
!pip install -q -U google-genai

In [23]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client ready!")

Gemini client ready!


In [24]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one short sentence."
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}

In [25]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one short sentence."
)

print(response.text)

Hello, it is great to meet you!


In [26]:
def generate_answer(query, retrieved_results):
    context = "\n\n".join(
        [
            f"Chunk {r['chunk_id']}:\n{r['content']}"
            for r in retrieved_results
        ]
    )

    prompt = f"""
You are a RAG question-answering assistant.

Answer the user's question ONLY using the provided context.

If the answer is not supported by the context, clearly say:
"Insufficient information in the retrieved context."

Do not use outside knowledge.
Do not invent facts.

Context:
{context}

Question:
{query}

Answer:
"""

    interaction = client.interactions.create(
        model="gemini-3.8-flash",
        input=prompt
    )

    return interaction.output_text


# Test with Query 1
query = test_queries[0]
retrieved = retrieve_chunks(query, top_k=3)

answer = generate_answer(query, retrieved)

print("QUERY:")
print(query)

print("\nGEMINI ANSWER:")
print(answer)

QUERY:
What is the main conclusion of this document?

GEMINI ANSWER:
Insufficient information in the retrieved context.


In [27]:
import json

diagnostic_results = []

for i, query in enumerate(test_queries, start=1):
    print(f"Processing Query {i}/15...")

    # Retrieve top 3 chunks
    retrieved = retrieve_chunks(query, top_k=3)

    # Generate grounded answer
    answer = generate_answer(query, retrieved)

    diagnostic_results.append({
        "query_id": i,
        "query": query,
        "retrieved_chunks": retrieved,
        "final_answer": answer
    })

print("\n✅ All 15 queries processed successfully!")
print("Total results:", len(diagnostic_results))

Processing Query 1/15...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded for model gemini-3.8-flash (limit: 20 requests per day on Free Tier). Please retry in 58s or upgrade your tier at https://ai.dev/rate-limit.', 'code': 'too_many_requests'}}

In [28]:
import time

print("Waiting 60 seconds for rate-limit window...")
time.sleep(60)

print("Trying Gemini again...")

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Say hello in one short sentence."
)

print(interaction.output_text)

Waiting 60 seconds for rate-limit window...
Trying Gemini again...
Hello, I hope you are having a wonderful day!


In [29]:
import json
import time

diagnostic_results = []

for i, query in enumerate(test_queries, start=1):
    print(f"\nProcessing Query {i}/15...")

    retrieved = retrieve_chunks(query, top_k=3)

    try:
        answer = generate_answer(query, retrieved)

        diagnostic_results.append({
            "query_id": i,
            "query": query,
            "retrieved_chunks": retrieved,
            "final_answer": answer
        })

        # Save after every successful query
        with open("diagnostic_results.json", "w", encoding="utf-8") as f:
            json.dump(diagnostic_results, f, indent=2, ensure_ascii=False)

        print("✅ Query completed")

        # Small pause between requests
        time.sleep(3)

    except Exception as e:
        print("⚠️ Gemini error:", e)
        print(f"Stopped after {i-1} successful queries.")
        break

print("\nSuccessful queries:", len(diagnostic_results))
print("Saved to: diagnostic_results.json")


Processing Query 1/15...
⚠️ Gemini error: Error code: 429 - {'error': {'message': 'Rate limit exceeded for model gemini-3.8-flash (limit: 20 requests per day on Free Tier). Please retry later or upgrade your tier at https://ai.dev/rate-limit.', 'code': 'too_many_requests'}}
Stopped after 0 successful queries.

Successful queries: 0
Saved to: diagnostic_results.json


In [30]:
def generate_answer(query, retrieved_results):
    context = "\n\n".join(
        [
            f"Chunk {r['chunk_id']}:\n{r['content']}"
            for r in retrieved_results
        ]
    )

    prompt = f"""
You are a RAG question-answering assistant.

Answer the user's question ONLY using the provided context.

If the answer is not supported by the context, say:
"Insufficient information in the retrieved context."

Do not use outside knowledge.
Do not invent facts.

Context:
{context}

Question:
{query}

Answer:
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return interaction.output_text


print("✅ generate_answer updated to Gemini 3.6 Flash")

✅ generate_answer updated to Gemini 3.6 Flash


In [31]:
query = test_queries[0]

retrieved = retrieve_chunks(query, top_k=3)

answer = generate_answer(query, retrieved)

print("QUERY:")
print(query)

print("\nGEMINI ANSWER:")
print(answer)

QUERY:
What is the main conclusion of this document?

GEMINI ANSWER:
Insufficient information in the retrieved context.


In [32]:
import json
import time

diagnostic_results = []

for i, query in enumerate(test_queries, start=1):
    print(f"\nProcessing Query {i}/15...")

    retrieved = retrieve_chunks(query, top_k=3)

    try:
        answer = generate_answer(query, retrieved)
        status = "success"

    except Exception as e:
        answer = "Gemini error: " + str(e)
        status = "error"

    diagnostic_results.append({
        "query_id": i,
        "query": query,
        "retrieved_chunks": retrieved,
        "final_answer": answer,
        "status": status
    })

    # Save after every query
    with open("diagnostic_results.json", "w", encoding="utf-8") as f:
        json.dump(
            diagnostic_results,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Status:", status)

    if status == "error":
        print("⚠️ Gemini request failed.")
        break

    time.sleep(3)

print("\n==============================")
print("Completed:", len(diagnostic_results), "/ 15")
print("Saved: diagnostic_results.json")
print("==============================")


Processing Query 1/15...
Status: success

Processing Query 2/15...
Status: success

Processing Query 3/15...
Status: success

Processing Query 4/15...
Status: success

Processing Query 5/15...
Status: success

Processing Query 6/15...
Status: success

Processing Query 7/15...
Status: error
⚠️ Gemini request failed.

Completed: 7 / 15
Saved: diagnostic_results.json


In [33]:
for item in diagnostic_results:
    print("\n" + "=" * 70)
    print(f"QUERY {item['query_id']}: {item['query']}")
    print(f"STATUS: {item['status']}")

    print("\nGEMINI ANSWER:")
    print(item["final_answer"])

    print("\nRETRIEVED CHUNKS:")
    for r in item["retrieved_chunks"]:
        print(
            f"Rank {r['rank']} | "
            f"Chunk {r['chunk_id']} | "
            f"Distance {r['distance']:.4f}"
        )
        print(r["content"][:250].replace("\n", " "))


QUERY 1: What is the main conclusion of this document?
STATUS: success

GEMINI ANSWER:
Insufficient information in the retrieved context.

RETRIEVED CHUNKS:
Rank 1 | Chunk 0 | Distance 1.6389
Unit – 1: Lec – 1 Today’s Target  Need of Programming Languages  Introduction of Python  First Program in Python  Identifiers, Keywords & Comments  Variables   AKTU PYQs Gateway Classes  Gateway Classes  UNIT-1: LEC-1 AKTU PYQs Q.1 : Write a Py
Rank 2 | Chunk 49 | Distance 1.6839
: Write short notes on the following with examples : AKTU(2024-25) a) Operator Precedence b) Python Indentation c) Type Conversion  Q.6 : Differentiate between / and // operator with an example . AKTU(2023-24) Q.7 Explain the input () function with s
Rank 3 | Chunk 54 | Distance 1.7167
s .   Given : a = 3  b = 3 Gateway Classes  OPERATORS PRECEDENCE & ASSOCIATIVITY OPERATOR PRECEDENCE : • Operator Precedence defines the order in which operators are evaluated in an expression . • Operators with higher precedence are

In [34]:
failure_analysis = [
    {
        "query_id": 1,
        "failure_type": "Retrieval Failure",
        "diagnosis": "The retrieved chunks do not contain the main conclusion of the document, so the required information was not retrieved."
    },
    {
        "query_id": 2,
        "failure_type": "Retrieval Failure",
        "diagnosis": "The query refers to a final topic/example that is not clearly represented in the retrieved chunks."
    },
    {
        "query_id": 3,
        "failure_type": "Correct Chunk Retrieved but Wrong Answer Generated",
        "diagnosis": "Chunk 11 contains the definition of identifiers, but the generated answer incorrectly says the information is insufficient."
    },
    {
        "query_id": 4,
        "failure_type": "Vague Context Retrieved",
        "diagnosis": "The vague question retrieves unrelated concepts such as identity operators, the Hello program, and relational operators."
    },
    {
        "query_id": 5,
        "failure_type": "Retrieval Failure",
        "diagnosis": "The retrieved chunks describe different Python topics but do not establish what happens immediately after a specific process."
    },
    {
        "query_id": 6,
        "failure_type": "Vague Context Retrieved",
        "diagnosis": "The query asks for steps of an unspecified process, while the retrieved chunks mainly describe operator precedence rather than a clearly defined process."
    },
    {
        "query_id": 7,
        "failure_type": "Vague Context Retrieved",
        "diagnosis": "The query is too broad to identify a specific information need, producing multiple unrelated chunks from the document."
    }
]

print("Failure Analysis\n")

for item in failure_analysis:
    print(f"Query {item['query_id']}")
    print(f"Failure Type: {item['failure_type']}")
    print(f"Diagnosis: {item['diagnosis']}")
    print("-" * 70)

Failure Analysis

Query 1
Failure Type: Retrieval Failure
Diagnosis: The retrieved chunks do not contain the main conclusion of the document, so the required information was not retrieved.
----------------------------------------------------------------------
Query 2
Failure Type: Retrieval Failure
Diagnosis: The query refers to a final topic/example that is not clearly represented in the retrieved chunks.
----------------------------------------------------------------------
Query 3
Failure Type: Correct Chunk Retrieved but Wrong Answer Generated
Diagnosis: Chunk 11 contains the definition of identifiers, but the generated answer incorrectly says the information is insufficient.
----------------------------------------------------------------------
Query 4
Failure Type: Vague Context Retrieved
Diagnosis: The vague question retrieves unrelated concepts such as identity operators, the Hello program, and relational operators.
--------------------------------------------------------------

In [35]:
scorecard = [
    {"query_id": 1, "retrieval_quality": 2, "answer_quality": 4},
    {"query_id": 2, "retrieval_quality": 2, "answer_quality": 4},
    {"query_id": 3, "retrieval_quality": 5, "answer_quality": 1},
    {"query_id": 4, "retrieval_quality": 2, "answer_quality": 4},
    {"query_id": 5, "retrieval_quality": 2, "answer_quality": 4},
    {"query_id": 6, "retrieval_quality": 2, "answer_quality": 4},
    {"query_id": 7, "retrieval_quality": 3, "answer_quality": 0}
]

avg_retrieval = sum(
    x["retrieval_quality"] for x in scorecard
) / len(scorecard)

avg_answer = sum(
    x["answer_quality"] for x in scorecard
) / len(scorecard)

print("RAG SCORECARD")
print("=" * 40)

for x in scorecard:
    print(
        f"Query {x['query_id']}: "
        f"Retrieval = {x['retrieval_quality']}/5, "
        f"Answer = {x['answer_quality']}/5"
    )

print("\nAverage Retrieval Quality:", round(avg_retrieval, 2), "/ 5")
print("Average Answer Quality:", round(avg_answer, 2), "/ 5")

RAG SCORECARD
Query 1: Retrieval = 2/5, Answer = 4/5
Query 2: Retrieval = 2/5, Answer = 4/5
Query 3: Retrieval = 5/5, Answer = 1/5
Query 4: Retrieval = 2/5, Answer = 4/5
Query 5: Retrieval = 2/5, Answer = 4/5
Query 6: Retrieval = 2/5, Answer = 4/5
Query 7: Retrieval = 3/5, Answer = 0/5

Average Retrieval Quality: 2.57 / 5
Average Answer Quality: 3.0 / 5


In [36]:
import numpy as np

def retrieve_with_threshold(query, top_k=3, threshold=0.35):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    results = []

    for idx, chunk in enumerate(chunks):
        chunk_embedding = model.encode(
            [chunk],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        similarity = float(np.dot(query_embedding, chunk_embedding))

        if similarity >= threshold:
            results.append({
                "chunk_id": idx,
                "similarity": similarity,
                "content": chunk
            })

    results.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return results[:top_k]


# Test
test_result = retrieve_with_threshold(
    "According to the document, what happened in 2025?"
)

print("Retrieved chunks:", len(test_result))

for r in test_result:
    print(
        f"\nChunk {r['chunk_id']} | "
        f"Similarity: {r['similarity']:.4f}"
    )
    print(r["content"][:300])

Retrieved chunks: 0


In [37]:
RETRIEVAL_THRESHOLD = 0.35

print("Fix #1: Similarity Threshold")
print("Threshold:", RETRIEVAL_THRESHOLD)
print("Purpose: Reject weak/irrelevant retrieved chunks")
print("Status: IMPLEMENTED ✅")

Fix #1: Similarity Threshold
Threshold: 0.35
Purpose: Reject weak/irrelevant retrieved chunks
Status: IMPLEMENTED ✅


In [38]:
def generate_grounded_answer(query, retrieved_results):
    if not retrieved_results:
        return "Insufficient information in the retrieved context."

    context = "\n\n".join(
        [
            f"Chunk {r['chunk_id']}:\n{r['content']}"
            for r in retrieved_results
        ]
    )

    prompt = f"""
You are a strict document-based RAG assistant.

Your job is to answer the question using ONLY the retrieved document context.

Rules:
1. Carefully read ALL retrieved chunks before answering.
2. If the answer is explicitly present in the context, answer it directly.
3. Do not say "insufficient information" when the context actually contains the answer.
4. If the answer is genuinely not present, say:
   "Insufficient information in the retrieved context."
5. Do not use outside knowledge.
6. Do not invent facts.
7. Keep the answer concise and directly related to the question.

Retrieved Context:
{context}

Question:
{query}

Answer:
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return interaction.output_text


print("Fix #2: Grounded Answer Prompt")
print("Status: IMPLEMENTED ✅")

Fix #2: Grounded Answer Prompt
Status: IMPLEMENTED ✅


In [39]:
query = test_queries[2]

retrieved = retrieve_chunks(query, top_k=3)

answer = generate_grounded_answer(query, retrieved)

print("QUERY:")
print(query)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED CHUNKS:")
for r in retrieved:
    print(
        f"\nChunk {r['chunk_id']} | "
        f"Distance: {r['distance']:.4f}"
    )
    print(r["content"][:400])

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded for model gemini-3.6-flash (limit: 20 requests per day on Free Tier). Please retry in 59s or upgrade your tier at https://ai.dev/rate-limit.', 'code': 'too_many_requests'}}

In [40]:
import json

with open("diagnostic_results.json", "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print("Saved diagnostic results:", len(saved_results))

for item in saved_results:
    print(
        f"Query {item['query_id']}: "
        f"{item['status']}"
    )

Saved diagnostic results: 7
Query 1: success
Query 2: success
Query 3: success
Query 4: success
Query 5: success
Query 6: success
Query 7: error


In [41]:
import json

# Existing 15-query retrieval results
master_results = []

for i, item in enumerate(all_results, start=1):
    master_results.append({
        "query_id": i,
        "query": item["query"],
        "retrieved_chunks": item["retrieved_chunks"]
    })

# Add Gemini answers where available
answer_lookup = {
    item["query_id"]: item["final_answer"]
    for item in diagnostic_results
}

status_lookup = {
    item["query_id"]: item["status"]
    for item in diagnostic_results
}

for item in master_results:
    qid = item["query_id"]

    if qid in answer_lookup:
        item["final_answer"] = answer_lookup[qid]
        item["gemini_status"] = status_lookup[qid]
    else:
        item["final_answer"] = "Not generated — Gemini rate limit reached."
        item["gemini_status"] = "not_run"

# Save complete master log
with open("day16_master_results.json", "w", encoding="utf-8") as f:
    json.dump(master_results, f, indent=2, ensure_ascii=False)

print("✅ Day 16 master log created")
print("Total queries:", len(master_results))
print("Queries with Gemini results:", len(answer_lookup))
print("Saved as: day16_master_results.json")

✅ Day 16 master log created
Total queries: 15
Queries with Gemini results: 7
Saved as: day16_master_results.json


In [42]:
import json

# Failure analysis already created
failure_lookup = {
    item["query_id"]: item
    for item in failure_analysis
}

# Build report
report = []

report.append("# Day 16 – Diagnosing RAG Failure Modes\n")

report.append("## 1. Objective\n")
report.append(
    "This experiment evaluates a Retrieval-Augmented Generation (RAG) "
    "pipeline across multiple failure modes including retrieval failure, "
    "vague context retrieval, and correct retrieval followed by incorrect "
    "answer generation.\n"
)

report.append("## 2. RAG Pipeline\n")
report.append(
    "PDF → Text Extraction → Chunking → Embeddings → FAISS Retrieval → "
    "Top-3 Retrieved Chunks → Gemini Answer Generation\n"
)

report.append("## 3. Dataset and Retrieval Setup\n")
report.append("- Document: Python programming lecture PDF\n")
report.append("- Chunk size: 500 characters\n")
report.append("- Chunk overlap: 100 characters\n")
report.append("- Embedding model: all-MiniLM-L6-v2\n")
report.append("- Vector database: FAISS IndexFlatL2\n")
report.append("- Retrieved chunks per query: Top 3\n")
report.append("- Total diagnostic queries: 15\n")

report.append("\n## 4. Failure Classification\n")

for item in failure_analysis:
    report.append(
        f"\n### Query {item['query_id']}\n"
        f"- **Failure Type:** {item['failure_type']}\n"
        f"- **Diagnosis:** {item['diagnosis']}\n"
    )

report.append("\n## 5. Important Observed Failure\n")
report.append(
    "Query 3 demonstrates a generation-stage failure. The retrieved "
    "Chunk 11 contains the definition of identifiers, but the generated "
    "answer incorrectly reported that the information was insufficient. "
    "This shows that correct retrieval does not always guarantee a correct answer.\n"
)

report.append("\n## 6. Fixes Implemented\n")

report.append(
    "\n### Fix 1 – Similarity Threshold\n"
    "- Added a cosine-similarity based retrieval threshold of 0.35.\n"
    "- Weakly related chunks are rejected instead of being passed to the answer generator.\n"
    "- Test query about events in 2025 returned 0 chunks at this threshold.\n"
)

report.append(
    "\n### Fix 2 – Grounded Answer Prompt\n"
    "- Added a strict grounded-answer prompt.\n"
    "- The model must use only retrieved document context.\n"
    "- Outside knowledge and invented facts are prohibited.\n"
    "- If the context genuinely does not contain the answer, the model must state that clearly.\n"
)

report.append("\n## 7. Scorecard\n")

report.append(
    "\n| Query | Retrieval Quality | Answer Quality |\n"
    "|---|---:|---:|\n"
)

for item in scorecard:
    report.append(
        f"| {item['query_id']} | "
        f"{item['retrieval_quality']}/5 | "
        f"{item['answer_quality']}/5 |\n"
    )

report.append(
    f"\n**Average Retrieval Quality:** {avg_retrieval:.2f}/5\n"
)

report.append(
    f"**Average Answer Quality:** {avg_answer:.2f}/5\n"
)

report.append("\n## 8. Gemini API Limitation\n")
report.append(
    "Gemini answer generation was successfully completed for the first "
    "six queries. Query 7 and subsequent generation attempts were affected "
    "by the Gemini Free Tier rate limit. Therefore, the experiment does not "
    "claim 15 successful Gemini-generated answers. Retrieval evaluation "
    "was completed for all 15 queries, while generation results are "
    "available only for the queries successfully processed before the limit.\n"
)

report.append("\n## 9. Conclusion\n")
report.append(
    "The experiment demonstrated that RAG failures can occur at multiple "
    "stages. Poor retrieval can provide irrelevant context, while correct "
    "retrieval can still be followed by an incorrect generated answer. "
    "A similarity threshold was implemented to reduce weak retrieval, and "
    "a grounded prompt was implemented to improve answer grounding.\n"
)

# Save report
with open("Day_16_RAG_Failure_Analysis.md", "w", encoding="utf-8") as f:
    f.writelines(report)

print("✅ FINAL DAY 16 REPORT CREATED")
print("File: Day_16_RAG_Failure_Analysis.md")

✅ FINAL DAY 16 REPORT CREATED
File: Day_16_RAG_Failure_Analysis.md


In [43]:
from google.colab import files

files.download("Day_16_RAG_Failure_Analysis.md")
files.download("day16_master_results.json")
files.download("diagnostic_results.json")
files.download("results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>